# 01 · Your first LLM call

**Course:** Building LLM Agents, notebook 1 of 10
**Where we are:** the very beginning. Before we can build *agents*, we need to be fluent with the raw material they are made of: a single call to a language model.

### What you'll learn
- How to connect the standard `openai` client to our Sigma2 GPT endpoint
- The anatomy of a chat completion: messages, roles, and the response object
- How to steer output with `temperature` and `max_tokens`
- Why LLMs are stateless, and how multi-turn chat actually works
- How to stream tokens as they're generated
- How to see which model your project can use, and how to read token usage

### Prerequisites
- You've run `pip install -r requirements.txt`
- You've copied `.env.example` to `.env` and pasted in your Sigma2 token

> Mental model: a large language model is, at heart, a very good *next-token predictor*. You give it a sequence of text (the conversation so far), and it predicts what comes next, one token at a time. Everything else in this course, including tools, memory, multiple agents, and orchestration, is scaffolding around that one ability. Keep that picture in mind. It explains almost every behaviour you'll see.

## 1. Connecting to the model

We talk to the model through the OpenAI Python SDK. That might sound odd when we're using a Sigma2 model, but "OpenAI-compatible" is an industry-standard *API shape*, and our Sigma2 AI Gateway (a [LiteLLM](https://litellm.ai/) proxy) speaks it. So the same client library works.

Two settings make the connection, both read from your `.env` file:

| Setting | Value | What it is |
|---|---|---|
| `OPENAI_API_KEY` | *your token* | proves who you are |
| `OPENAI_BASE_URL` | `https://ai-gateway.sigma2.no/v1` | *where* to send requests |

In every later notebook we'll just write `from helpers import client, MODEL`. But this first time, let's build the client by hand so you can see exactly what that helper is doing.

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI

# find_dotenv() walks up the folders until it finds the .env at the repo root,
# so this works no matter which directory you launched Jupyter from.
load_dotenv(find_dotenv())

client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"],
    base_url=os.environ["OPENAI_BASE_URL"],
)

### Which model are we using?

Your project on the endpoint has been set up with one or more models, a bit like ordering a service for your team and getting access to exactly what was provisioned for you. So instead of hard-coding a model name, let's ask the endpoint what you can use:

In [ ]:
available = [m.id for m in client.models.list().data]
print("Models available to your project:")
for mid in available:
    print("  -", mid)

We'll use the first *chat* model for the rest of the course. Your key may also list `clef-flash-not-for-production`: that's Clef, a *decision* model that answers questions with probabilities instead of writing text. It can't chat, so we skip it here; notebook 11 is about it.

To pin a specific model, set `COURSE_MODEL` in your `.env`. That's exactly what `helpers.py` checks first.

In [ ]:
# Skip the models that can't chat: "arena-model" is a built-in comparison entry,
# and Clef ("clef-...") is a decision model you'll meet in notebook 11.
MODEL = next(m for m in available if m != "arena-model" and not m.startswith("clef"))
print("Using model:", MODEL)

Let's confirm the connection is wired up, without printing your secret token to the screen.

In [ ]:
key = os.environ["OPENAI_API_KEY"]
print("Base URL :", os.environ["OPENAI_BASE_URL"])
print("Model    :", MODEL)
print("Token    :", "…" + key[-4:], f"({len(key)} chars)")

## 2. Your first completion

The core operation is `client.chat.completions.create(...)`. You pass it:

- a `model` name, and
- a list of `messages`, the conversation so far.

Each message is a dict with a `role` and `content`. The simplest possible call is one `user` message:

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "In one sentence, what is a large language model?"}
    ],
)

print(response.choices[0].message.content)

That's a real LLM call to a supercomputer in Norway.

### What came back
The response is a structured object, not just text. The parts you'll use constantly:

- `response.choices[0].message` is the reply (`.role` and `.content`)
- `response.choices[0].finish_reason` says *why* it stopped: `"stop"` means it finished naturally, `"length"` means it hit the token limit, and `"tool_calls"` means it wants to use a tool. That last one becomes very important from notebook 03.
- `response.usage` reports how many tokens were used

Let's look at the skeleton:

![Anatomy of a chat-completion request and response](figures/01-chat-anatomy.svg)

*Figure: the request to response round trip, and the parts of the response object you use. Generate it from the prompt in [`figures/FIGURES.md`](figures/FIGURES.md).*

In [ ]:
msg = response.choices[0].message
print("role          :", msg.role)
print("finish_reason :", response.choices[0].finish_reason)
print("usage         :", response.usage)

## 3. Roles: `system`, `user`, `assistant`

A conversation is a list of messages, and each has a role:

- `system` sets standing instructions that shape *how* the model behaves: its persona, rules, and format. It comes first.
- `user` is what the human says.
- `assistant` is what the model said previously (we'll add these ourselves for multi-turn chat in section 5).

The `system` message is your main lever for controlling behaviour. Watch the same question get a very different answer when we change it:

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a grumpy pirate. Answer in one short sentence."},
        {"role": "user", "content": "What is a large language model?"},
    ],
)
print(response.choices[0].message.content)

## 4. Controlling the output: `temperature` and `max_tokens`

Two knobs you'll reach for all the time:

- `temperature` (0.0 to about 1.5) controls randomness. `0` is almost deterministic and focused, which is best for tools, extraction, and anything you want to be reliable. Higher values are more creative and varied.
- `max_tokens` is a hard cap on how long the reply can be. It's useful for cost control and for keeping answers tight.

Run the next cell a couple of times. At `temperature=0` you'll get nearly the same line each time. Bump it to `1.0` and the outputs start to vary.

In [ ]:
def slogan(temperature):
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": "Write a 6-word slogan for a coffee shop."}],
        temperature=temperature,
        max_tokens=8000,
    )
    return r.choices[0].message.content.strip()

print("temp=0.0 :", slogan(0.0))
print("temp=1.0 :", slogan(1.0))
print("temp=1.0 :", slogan(1.0))

> Rule of thumb for this course: when we build agents, we'll almost always use `temperature=0`. An agent deciding *which tool to call* should be predictable, not creative.

![Stateless model: a conversation is the growing message list you resend each call](figures/01-stateless-memory.svg)

*Figure: the model keeps no state of its own. Its "memory" is the message list you resend every turn. Generate it from the prompt in [`figures/FIGURES.md`](figures/FIGURES.md).*

## 5. Multi-turn conversations (and why the model has no memory)

Here's the single most important practical fact about LLMs:

> **The model is stateless.** It remembers *nothing* between calls. The only thing it "knows" is the list of messages you send in *this* request.

So a "conversation" is an illusion that **you** maintain: each turn, you send the *entire history*, including the model's own previous replies as `assistant` messages, and then append the new `user` message.

Let's do it manually so the mechanism is clear:

In [ ]:
# We keep growing this list ourselves. THIS is the conversation's memory.
messages = [
    {"role": "system", "content": "You are a concise geography tutor."},
    {"role": "user", "content": "What is the capital of Norway?"},
]

r1 = client.chat.completions.create(model=MODEL, messages=messages, temperature=0)
answer1 = r1.choices[0].message.content
print("Assistant:", answer1)

# Append the model's reply, then ask a follow-up that only makes sense *with* history.
messages.append({"role": "assistant", "content": answer1})
messages.append({"role": "user", "content": "And what is its population, roughly?"})

r2 = client.chat.completions.create(model=MODEL, messages=messages, temperature=0)
print("Assistant:", r2.choices[0].message.content)

The model answered the follow-up ("its population") correctly *only* because we resent the whole history. Otherwise it had no idea what "it" referred to.

Let's view the full conversation with a little helper from `helpers.py`. We'll lean on it a lot once message lists get long:

In [ ]:
from helpers import show_messages
show_messages(messages)

> Foreshadowing: "managing the list of messages you send" is exactly what memory (notebook 05) is about, and the growing history is what eventually bumps into the context window limit. File that away.

## 6. Streaming

By default you wait for the whole answer, then get it all at once. With `stream=True` the model sends tokens as it produces them, which is how chat UIs show text appearing live. You get an iterator of *chunks*, and the new text is in `chunk.choices[0].delta.content`.

In [ ]:
stream = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Count from 1 to 10, with a short pause word between each."}],
    stream=True,
)

for chunk in stream:
    piece = chunk.choices[0].delta.content
    if piece:
        print(piece, end="", flush=True)
print()  # final newline

## 7. Getting the model to reason

On harder questions the answer quality jumps when you let the model work through it step by step instead of demanding a one-shot answer. This isn't a special feature of any particular model. It's just prompting, and it works everywhere. Compare the two styles:

In [ ]:
q = "A train leaves at 14:00 and the trip takes 95 minutes. When does it arrive?"

# Style A: demand the answer immediately
direct = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": q + " Reply with only the arrival time."}],
    temperature=0, max_tokens=8000,
)
print("DIRECT:", direct.choices[0].message.content)

# Style B: ask it to think first, then answer
reasoned = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": q + " Think step by step, then give the arrival time."}],
    temperature=0, max_tokens=8000,
)
print("\nREASONED:\n", reasoned.choices[0].message.content)

> Why you care: getting the model to reason *before* it commits to an answer is the heartbeat of an agent. In notebook 04 the agent will reason about *which tool to use* before acting. "Think step by step" is the simplest version of a technique we'll lean on all course long.

## 8. Tokens and cost

Models don't read characters. They read tokens (chunks of text, roughly three-quarters of a word in English). Every request reports its usage, and on a shared cluster like Sigma2 that's your "spend." Get in the habit of glancing at it.

In [ ]:
r = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "List three Norwegian cities."}],
    temperature=0,
)
u = r.usage
print(f"prompt tokens     : {u.prompt_tokens}")
print(f"completion tokens : {u.completion_tokens}")
print(f"total tokens      : {u.total_tokens}")

> The agent cost trap: a single chat call is cheap. But an *agent* may loop. It calls the model, uses a tool, then calls the model again, maybe ten times over, and each loop resends the whole growing history. Token usage can balloon fast. We'll come back to this in the capstone (notebook 10).

## Recap

You can now:
- connect the `openai` client to the Sigma2 endpoint
- send a chat completion and read the response object
- use `system` / `user` / `assistant` roles and steer with `temperature` / `max_tokens`
- maintain a multi-turn conversation by resending history (the model is stateless)
- list the models your project can use, stream tokens, prompt for step-by-step reasoning, and read token usage

### Try it yourself
1. Give the system prompt a personality of your choice and re-run section 3.
2. Set `max_tokens=5` on a long question. What `finish_reason` do you get?
3. Extend the conversation in section 5 with a third follow-up question.
4. Stream a short story and watch it appear word by word.

### Next up
**02 · Prompting & structured output:** how to write prompts that behave reliably, and how to get the model to return clean JSON your code can use. That's the bridge from "chatbot" to "building block."